# Flappy Bird: Double Dueling DQN
Hai chế độ quan sát: `pixel` (CNN) và `state` (MLP). Mã nguồn nằm trong `flappy_rl/`.

In [ ]:
import subprocess, sys, zipfile
from pathlib import Path

try:
    import flappy_rl
except ImportError:  # chạy trên Colab: giải nén dự án từ Drive rồi cài
    from google.colab import drive
    drive.mount("/content/drive")
    with zipfile.ZipFile("/content/drive/MyDrive/FlappyBird/flappy_bird.zip") as z:
        z.extractall("/content/flappy_bird")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", "/content/flappy_bird"])

## Cấu hình

In [ ]:
from datetime import datetime
import torch
from flappy_rl import AgentConfig, DQNAgent, TrainConfig
from flappy_rl.envs import make_env
from flappy_rl.runtime import resolve_device, set_seed

MODE = "pixel"          # "pixel" hoặc "state"
RESUME = ""             # đường dẫn checkpoint để train tiếp (để trống nếu train mới)
DEVICE = resolve_device("auto")
SEED = 42

run_dir = Path("runs") / f"{MODE}_{datetime.now():%Y%m%d_%H%M%S}"
run_dir.mkdir(parents=True)
train_cfg = TrainConfig(seed=SEED, best_path=str(run_dir / "best.pt"), last_path=str(run_dir / "last.pt"))

set_seed(SEED, DEVICE)
if RESUME:
    agent = DQNAgent.from_checkpoint(RESUME, DEVICE, training=True)
    MODE = agent.config.mode
else:
    agent = DQNAgent(AgentConfig.for_mode(MODE), DEVICE)
print(f"{DEVICE} | {MODE} | replay {agent.memory.nbytes / 1024**3:.2f} GiB")

from flappy_rl.training import train

## Huấn luyện
Checkpoint tốt nhất được chọn theo điểm greedy trên các seed cố định.

In [ ]:
env, eval_env = make_env(MODE), make_env(MODE)
try:
    history = train(agent, env, eval_env, train_cfg)
finally:
    env.close(); eval_env.close()
history.save(run_dir / "history.json")

In [ ]:
from flappy_rl.plotting import plot_training
plot_training(history, out_path=run_dir / "training_results.png")

## Đánh giá
Điểm holdout dùng seed khác với seed chọn checkpoint.

In [ ]:
from IPython.display import Video, display
from flappy_rl.evaluation import evaluate_policy, record_video

best = DQNAgent.from_checkpoint(train_cfg.best_path, DEVICE)
holdout_env = make_env(MODE)
try:
    print("Holdout:", evaluate_policy(best, holdout_env, num_episodes=10, max_steps=5000, seed=SEED + 20_000))
finally:
    holdout_env.close()
record_video(best, run_dir / "gameplay.mp4")
display(Video(str(run_dir / "gameplay.mp4"), embed=True))

## Giao diện 3D
Replay chạy cho cả hai chế độ; chạy mạng trực tiếp trong trình duyệt chỉ hỗ trợ `state`.

In [ ]:
from flappy_rl.export import export_mlp_weights, record_replay
from flappy_rl.viewer import build_3d_viewer, show_3d

replay = record_replay(best, out_path=run_dir / "replay.json")
weights = export_mlp_weights(best, out_path=run_dir / "weights.json")
display(show_3d(build_3d_viewer(replay, weights, out_path=run_dir / "viewer.html")))